In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from rgz import classifications
from rgz import plot_classifications
from rgz import subjects

In [ ]:
# Paths 
testdata_path = Path("rgz") / "testdata"
raw_subjects_path = testdata_path / "radio_subjects_test_subset.json"
processed_subjects_path = testdata_path / "radio_subjects_test_subset_processed.json"
processed_classifications_path = testdata_path / "radio_classifications_test_subset_processed.json"

In [ ]:
# For picking a subject based on what the IR/radio looks like
for subject_idx in range(31):
    with open(processed_subjects_path, "r") as f:
        subject_json = json.load(f)[subject_idx]
    subject = subjects.Subject.from_json(subject_json)
    print(str(subject_idx) + " " + subject.id)

In [ ]:
# Load a subject 
subject_idx = 11
with open(processed_subjects_path, "r") as f:
    subject_json = json.load(f)[subject_idx]
subject = subjects.Subject.from_json(subject_json)
print(subject.id)

# Find a classification that references this subject 
with open(processed_classifications_path, "r") as f:
    classifications_json = [c for c in json.load(f) if c["zid"] == subject.zid]
classifications_list = [classifications.Classification.from_json(cs) for cs in classifications_json]


In [ ]:
# Test general functionality
for ii in range(len(classifications_list))[:1]:
    ax = plot_classifications.plot_single_classification(classification=classifications_list[ii], 
                                            subject=subject,
                                            cache=testdata_path / "first",
                                            ax=None)

In [ ]:
# Test that existing axes are destroyed 
fig, ax = plt.subplots()
fig.subplots_adjust(left=0.1, right=0.3, top=0.7, bottom=0.05)
_ = plot_classifications.plot_single_classification(
    classification=classifications_list[0], 
    subject=subject,
    cache=testdata_path / "first",
    ax=ax,)


In [ ]:
# Should look like the FIRST/WISE images
contour_coords_list = plot_classifications.get_contours(
    subject=subject,
    px_coords=True,
    px_scaling=1,
    cache=testdata_path / "first",
)
zeroth_contour_coords_list = [c[0] for c in contour_coords_list]
fig, ax = plt.subplots()
for contour_coords in zeroth_contour_coords_list:
    ax.plot(
        *zip(*contour_coords),
        color="k",
    )

# Should be flipped left-to-right since RA increases to the left 
contour_coords_list = plot_classifications.get_contours(
    subject=subject,
    px_coords=False,
    px_scaling=1,
    cache=testdata_path / "first",
)
zeroth_contour_coords_list = [c[0] for c in contour_coords_list]
fig, ax = plt.subplots()
for contour_coords in zeroth_contour_coords_list:
    ax.plot(
        *zip(*contour_coords),
        color="k",
    )

In [ ]:
from astropy.coordinates import SkyCoord
from astropy.wcs import WCS
import matplotlib.pyplot as plt
import numpy as np

from rgz import cutouts



# Plot the AllWISE cutout for NGC1068
coords = SkyCoord(
    ra="02:42:40.71", dec="-00:00:47.86", unit=(u.hourangle, u.deg), equinox="J2000"
)
hdulist = cutouts.get_allwise_cutout(coords=coords, size=3.5 * u.arcmin)
im = hdulist[0].data
wcs = WCS(hdulist[0].header)
fig, ax = plt.subplots(subplot_kw=dict(projection=wcs))
ax.imshow(np.log10(im))
